In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
          'axes.labelsize': 'large',
          'axes.titlesize': 'large',
          'xtick.labelsize': 'large',
          'ytick.labelsize': 'large',
          'figure.facecolor': 'w',
          'xtick.top': True,
          'ytick.right': True,
          'xtick.direction': 'in',
          'ytick.direction': 'in',
         }
plt.rcParams.update(params)

In [3]:
# perexp = Table(fitsio.read('/global/cfs/cdirs/desi/users/rongpu/spectro/fugu/sv1_perexp_lrg.fits'))
perexp = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/sv1_perexp_lrg.fits'))
perexp['EFFTIME_ELG'] = 8.60 * perexp['TSNR2_ELG']
perexp['EFFTIME_LRG'] = 12.15 * perexp['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = perexp['FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
perexp = perexp[mask]

# Remove "no data" fibers
mask = perexp['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
perexp = perexp[mask]

# Apply LRG mask
mask = perexp['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
perexp = perexp[mask]

# Remove QSO targets
mask = perexp['SV1_DESI_TARGET'] & 2**2 ==0
print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
perexp = perexp[mask]

# Remove objects classified as QSOs
mask = perexp['SPECTYPE']!='QSO'
print('Remove objects classified as QSOs:', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
perexp = perexp[mask]

# Remove objects classified as STARs
mask = perexp['SPECTYPE']!='STAR'
print('Remove objects classified as STARs:', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
perexp = perexp[mask]

# # Require a minimum depth
# min_depth = 500.
# mask = perexp['EFFTIME_LRG']>min_depth
# print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
# perexp = perexp[mask]

# # Julien's bad fibers list + my list of worst-performing fibers; bad_fibers-everest.ipynb
# # bad_fibers = np.loadtxt('/global/cfs/cdirs/desi/users/rongpu/spectro/everest/misc/bad_fibers_20211117.txt', dtype=int)
# bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211117.txt', dtype=int)
# print(len(bad_fibers))
# mask_bad = np.in1d(perexp['FIBER'], bad_fibers)
# print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
# perexp = perexp[~mask_bad]
# print(len(perexp), len(np.unique(perexp['TARGETID'])))

# Redshift quality cut
perexp['q'] = perexp['ZWARN']==0
perexp['q'] &= (perexp['Z']<1.5)
perexp['q'] &= perexp['DELTACHI2']>15

FIBERSTATUS 464751 107007 0.18715435551404616
No data 464751 0 0.0
LRG mask 417693 47058 0.10125422000167832
Remove QSO targets 403772 13921 0.03332830571735701
Remove objects classified as QSOs: 398156 5616 0.013908839642174296
Remove objects classified as STARs: 395487 2669 0.006703402686384231


In [4]:
# cat_1x = Table(fitsio.read('/global/cfs/cdirs/desi/users/rongpu/spectro/fugu/sv1_1x_depth_lrg.fits'))
cat_1x = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/sv1_1x_depth_lrg.fits'))
cat_1x['EFFTIME_ELG'] = 8.60 * cat_1x['TSNR2_ELG']
cat_1x['EFFTIME_LRG'] = 12.15 * cat_1x['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat_1x['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[mask]

# Remove "no data" fibers
mask = cat_1x['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[mask]

# Apply LRG mask
mask = cat_1x['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[mask]

# Remove QSO targets
mask = cat_1x['SV1_DESI_TARGET'] & 2**2 ==0
print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[mask]

# Remove objects classified as QSOs
mask = cat_1x['SPECTYPE']!='QSO'
print('Remove objects classified as QSOs:', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[mask]

# Remove objects classified as STARs
mask = cat_1x['SPECTYPE']!='STAR'
print('Remove objects classified as STARs:', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[mask]

# # Require a minimum depth
# min_depth = 500.
# mask = cat_1x['EFFTIME_LRG']>min_depth
# print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
# cat_1x = cat_1x[mask]

# # Julien's bad fibers list + my list of worst-performing fibers; bad_fibers-everest.ipynb
# # bad_fibers = np.loadtxt('/global/cfs/cdirs/desi/users/rongpu/spectro/everest/misc/bad_fibers_20211117.txt', dtype=int)
# bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211117.txt', dtype=int)
# print(len(bad_fibers))
# mask_bad = np.in1d(cat_1x['FIBER'], bad_fibers)
# print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
# cat_1x = cat_1x[~mask_bad]
# print(len(cat_1x), len(np.unique(cat_1x['TARGETID'])))

# Redshift quality cut
cat_1x['q'] = cat_1x['ZWARN']==0
cat_1x['q'] &= (cat_1x['Z']<1.5)
cat_1x['q'] &= cat_1x['DELTACHI2']>15

FIBERSTATUS 27110 4943 0.15421333416528873
No data 27110 0 0.0
LRG mask 25008 2102 0.07753596458871265
Remove QSO targets 24493 515 0.020593410108765194
Remove objects classified as QSOs: 24169 324 0.013228269301433063
Remove objects classified as STARs: 24013 156 0.0064545492159377715


In [5]:
cat = vstack([perexp, cat_1x], join_type='inner')
print(len(cat))

419500


In [6]:
# deep = Table(fitsio.read('/global/cfs/cdirs/desi/users/rongpu/spectro/fugu/sv1_cumulative_lrg.fits'))
deep = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/sv1_cumulative_lrg.fits'))
deep['EFFTIME_ELG'] = 8.60 * deep['TSNR2_ELG']
deep['EFFTIME_LRG'] = 12.15 * deep['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = deep['COADD_FIBERSTATUS']==0
print('COADD_FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
deep = deep[mask]

mask = deep['ZWARN']==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
deep = deep[mask]

# Apply LRG mask
mask = deep['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
deep = deep[mask]

# # Remove QSO targets
# mask = deep['SV1_DESI_TARGET'] & 2**2 ==0
# print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
# deep = deep[mask]

# Remove objects classified as QSOs
mask = deep['SPECTYPE']!='QSO'
print('Remove objects classified as QSOs:', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
deep = deep[mask]

# Remove objects classified as STARs
mask = deep['SPECTYPE']!='STAR'
print('Remove objects classified as STARs:', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
deep = deep[mask]

# # Julien's bad fibers list + my list of worst-performing fibers; bad_fibers-everest.ipynb
# # bad_fibers = np.loadtxt('/global/cfs/cdirs/desi/users/rongpu/spectro/everest/misc/bad_fibers_20211117.txt', dtype=int)
# bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211117.txt', dtype=int)
# print(len(bad_fibers))
# mask_bad = np.in1d(deep['FIBER'], bad_fibers)
# print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
# deep = deep[~mask_bad]
# print(len(deep), len(np.unique(deep['TARGETID'])))

# Remove duplidates keeping the higher EFFTIME objects
deep.sort('EFFTIME_LRG', reverse=True)
_, idx_keep = np.unique(deep['TARGETID'], return_index=True)
deep = deep[idx_keep]
print(len(deep), len(np.unique(deep['TARGETID'])))

#####################################################################################################################
deep_tmp = deep.copy()
# Require a minimum depth
min_depth = 3000.
mask = deep_tmp['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
deep_tmp = deep_tmp[mask]

deep_columns_old = ['TARGETID', 'Z', 'ZERR', 'ZWARN', 'SPECTYPE', 'DELTACHI2', 'EFFTIME_LRG', 'EFFTIME_ELG']
deep_columns_new = ['TARGETID', 'Z_deep', 'ZERR_deep', 'ZWARN_deep', 'SPECTYPE_deep', 'DELTACHI2_deep', 'EFFTIME_LRG_deep', 'EFFTIME_ELG_deep']
deep_tmp.rename_columns(deep_columns_old, deep_columns_new)

cat = join(cat, deep_tmp[deep_columns_new], keys='TARGETID', join_type='inner')
print(len(cat))
#####################################################################################################################

# Redshift quality cut
deep['q'] = deep['ZWARN']==0
deep['q'] &= (deep['Z']<1.5)
deep['q'] &= deep['DELTACHI2']>15

COADD_FIBERSTATUS 46319 8916 0.16141938987960533
No data 45437 882 0.01904186187093849
LRG mask 41340 4097 0.09016880515879129
Remove objects classified as QSOs: 40125 1215 0.02939042089985486
Remove objects classified as STARs: 39668 457 0.011389408099688473
39533 39533
Min depth 30355 9178 0.7678395264715554
367504


In [7]:
# Catastrophic redshift failures
zdiff_threshold = 0.0033
mask_fail = np.abs((cat['Z'] - cat['Z_deep'])/(1 + cat['Z_deep'])) > zdiff_threshold
print(np.sum(mask_fail), np.sum(mask_fail)/len(mask_fail))
# Also reject objects with ZWARN!=0 or z>1.5
mask_deep = (cat['ZWARN_deep']==0) & (cat['Z_deep']<1.5)
mask_deep_fail = ~mask_deep
# mask_deep = (cat['ZWARN_deep']==0) & (cat['Z_deep']<1.5) & (cat['DELTACHI2_deep']>30)
mask_fail |= mask_deep_fail
print(np.sum(mask_fail), np.sum(mask_fail)/len(mask_fail))
print()

print('Main LRGs in SV:')
mask = cat['main_lrg'].copy()
print('Catastrophic failure rate (all): {:.1f}% ({}/{})'.format(100*np.sum(mask_fail & mask)/np.sum(mask), np.sum(mask_fail & mask), np.sum(mask)))
mask &= (cat['EFFTIME_LRG']>800) & (cat['EFFTIME_LRG']<1200)
print('Catastrophic failure rate (800s<EFFTIME<1200s): {:.2f}% ({}/{})'.format(100*np.sum(mask_fail & mask)/np.sum(mask), np.sum(mask_fail & mask), np.sum(mask)))

67317 0.1831735164787322
67633 0.18403337106534895

Main LRGs in SV:
Catastrophic failure rate (all): 10.4% (10637/102586)
Catastrophic failure rate (800s<EFFTIME<1200s): 0.72% (110/15379)


In [8]:
print('Quality cuts\n')
print(np.sum(~cat['q']), np.sum(~cat['q'])/len(cat))
mask = (cat['EFFTIME_LRG']>800) & (cat['EFFTIME_LRG']<1200)
# Failures in "good redshifts"
print(np.sum(mask_fail & cat['q'] & mask), np.sum(mask_fail & cat['q'] & mask)/np.sum(cat['q'] & mask))
print()

print('Main LRGs in SV:')
mask = cat['main_lrg'].copy()
print('Rejection rate (all): {:.1f}% ({}/{})'.format(100*np.sum((~cat['q']) & mask)/np.sum(mask), np.sum((~cat['q']) & mask), np.sum(mask)))
mask &= (cat['EFFTIME_LRG']>800) & (cat['EFFTIME_LRG']<1200)
print('Rejection rate (800s<EFFTIME<1200s): {:.2f}% ({}/{})'.format(100*np.sum((~cat['q']) & mask)/np.sum(mask), np.sum((~cat['q']) & mask), np.sum(mask)))
print('Catastrophic failure rate in accepted redshifts (800s<EFFTIME<1200s): {:.2f}% ({}/{})'.format(100*np.sum(mask_fail & cat['q'] & mask)/np.sum(cat['q'] & mask), np.sum(mask_fail & cat['q'] & mask), np.sum(cat['q'] & mask)))
print('Catastrophic failure rate in rejected redshifts (800s<EFFTIME<1200s): {:.2f}% ({}/{})'.format(100*np.sum(mask_fail & (~cat['q']) & mask)/np.sum((~cat['q']) & mask), np.sum(mask_fail & (~cat['q']) & mask), np.sum((~cat['q']) & mask)))

Quality cuts

111116 0.3023531716661587
174 0.0034198785353485722

Main LRGs in SV:
Rejection rate (all): 17.7% (18134/102586)
Rejection rate (800s<EFFTIME<1200s): 1.24% (191/15379)
Catastrophic failure rate in accepted redshifts (800s<EFFTIME<1200s): 0.18% (28/15188)
Catastrophic failure rate in rejected redshifts (800s<EFFTIME<1200s): 42.93% (82/191)


-------
# compare with VI redshifts

In [10]:
# vi = Table.read('/global/cfs/cdirs/desi/sv/vi/TruthTables/Fuji/LRG/220505_LRG_SV1_blanc_and_Fuji_v1.csv')
vi = Table.read('/Users/rongpu/Documents/Data/desi_targets/lrg_paper/220505_LRG_SV1_blanc_and_Fuji_v1.csv')

In [11]:
print(len(cat), len(np.unique(cat['TARGETID'])))
cat = join(cat, vi[['TARGETID', 'best_z', 'best_quality', 'VI_version']], keys='TARGETID', join_type='left')
print(len(cat), len(np.unique(cat['TARGETID'])))
cat = cat.filled(-99)

367504 29770
367504 29770


In [32]:
mask = (cat['EFFTIME_LRG']>800) & (cat['EFFTIME_LRG']<1200)
mask &= (cat['best_quality']>=2.5)
print(np.sum(mask))
print()
mask_fail = np.abs((cat['Z'] - cat['Z_deep'])/(1 + cat['Z_deep'])) > zdiff_threshold
print('Failures in SV LRGs:')
print('{:.2f}% ({}/{})'.format(100*np.sum(mask & mask_fail)/np.sum(mask), np.sum(mask & mask_fail), np.sum(mask)))
mask_fail |= np.abs((cat['best_z'] - cat['Z'])/(1 + cat['Z'])) > zdiff_threshold
print('Failures in SV LRGs (including VI disagreements):')
print('{:.2f}% ({}/{})'.format(100*np.sum(mask & mask_fail)/np.sum(mask), np.sum(mask & mask_fail), np.sum(mask)))

mask = (cat['EFFTIME_LRG']>800) & (cat['EFFTIME_LRG']<1200)
mask &= (cat['best_quality']>=2.5)
mask &= cat['q']
print(np.sum(mask))
print()
mask_fail = np.abs((cat['Z'] - cat['Z_deep'])/(1 + cat['Z_deep'])) > zdiff_threshold
print('Failures in SV LRGs after quality cuts:')
print('{:.2f}% ({}/{})'.format(100*np.sum(mask & mask_fail)/np.sum(mask), np.sum(mask & mask_fail), np.sum(mask)))
mask_fail |= np.abs((cat['best_z'] - cat['Z'])/(1 + cat['Z'])) > zdiff_threshold
print('Failures in SV LRGs after quality cuts (including VI disagreements):')
print('{:.2f}% ({}/{})'.format(100*np.sum(mask & mask_fail)/np.sum(mask), np.sum(mask & mask_fail), np.sum(mask)))

18073

Failures in SV LRGs:
3.25% (588/18073)
Failures in SV LRGs (including VI disagreements):
3.51% (635/18073)
16689

Failures in SV LRGs after quality cuts:
0.50% (83/16689)
Failures in SV LRGs after quality cuts (including VI disagreements):
0.67% (111/16689)


In [33]:
mask = (cat['EFFTIME_LRG']>800) & (cat['EFFTIME_LRG']<1200)
mask &= (cat['best_quality']>=2.5)
mask &= cat['main_lrg'].copy()
print(np.sum(mask))
print()
mask_fail = np.abs((cat['Z'] - cat['Z_deep'])/(1 + cat['Z_deep'])) > zdiff_threshold
print('Failures in Main LRGs:')
print('{:.2f}% ({}/{})'.format(100*np.sum(mask & mask_fail)/np.sum(mask), np.sum(mask & mask_fail), np.sum(mask)))
mask_fail |= np.abs((cat['best_z'] - cat['Z'])/(1 + cat['Z'])) > zdiff_threshold
print('Failures in Main LRGs (including VI disagreements):')
print('{:.2f}% ({}/{})'.format(100*np.sum(mask & mask_fail)/np.sum(mask), np.sum(mask & mask_fail), np.sum(mask)))

mask = (cat['EFFTIME_LRG']>800) & (cat['EFFTIME_LRG']<1200)
mask &= (cat['best_quality']>=2.5)
mask &= cat['main_lrg'].copy()
mask &= cat['q']
print(np.sum(mask))
print()
mask_fail = np.abs((cat['Z'] - cat['Z_deep'])/(1 + cat['Z_deep'])) > zdiff_threshold
print('Failures in Main LRGs after quality cuts:')
print('{:.2f}% ({}/{})'.format(100*np.sum(mask & mask_fail)/np.sum(mask), np.sum(mask & mask_fail), np.sum(mask)))
mask_fail |= np.abs((cat['best_z'] - cat['Z'])/(1 + cat['Z'])) > zdiff_threshold
print('Failures in Main LRGs after quality cuts (including VI disagreements):')
print('{:.2f}% ({}/{})'.format(100*np.sum(mask & mask_fail)/np.sum(mask), np.sum(mask & mask_fail), np.sum(mask)))

4734

Failures in Main LRGs:
0.87% (41/4734)
Failures in Main LRGs (including VI disagreements):
0.93% (44/4734)
4664

Failures in Main LRGs after quality cuts:
0.28% (13/4664)
Failures in Main LRGs after quality cuts (including VI disagreements):
0.28% (13/4664)
